# 04 - Vantagem de Jogar em Casa

A vantagem do mandante está diminuindo ao longo dos anos? Como a pandemia afetou?

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv("../data/raw/campeonato-brasileiro-full.csv")
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
df["ano"] = df["data"].dt.year
df = df[df["ano"] >= 2003].copy()

df["resultado"] = df.apply(
    lambda r: "Mandante" if r["mandante_Placar"] > r["visitante_Placar"]
    else ("Visitante" if r["mandante_Placar"] < r["visitante_Placar"] else "Empate"),
    axis=1
)

In [2]:
# Percentual de resultados por ano
res_por_ano = df.groupby(["ano", "resultado"]).size().unstack(fill_value=0)
res_pct = res_por_ano.div(res_por_ano.sum(axis=1), axis=0) * 100
res_pct = res_pct.reset_index()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=res_pct["ano"], y=res_pct["Mandante"],
    mode="lines+markers", name="Vitória Mandante",
    line=dict(color="#27ae60", width=3),
    hovertemplate="%{x}: %{y:.1f}%<extra>Mandante</extra>"
))
fig.add_trace(go.Scatter(
    x=res_pct["ano"], y=res_pct["Empate"],
    mode="lines+markers", name="Empate",
    line=dict(color="#f39c12", width=3),
    hovertemplate="%{x}: %{y:.1f}%<extra>Empate</extra>"
))
fig.add_trace(go.Scatter(
    x=res_pct["ano"], y=res_pct["Visitante"],
    mode="lines+markers", name="Vitória Visitante",
    line=dict(color="#e74c3c", width=3),
    hovertemplate="%{x}: %{y:.1f}%<extra>Visitante</extra>"
))

# Destacar 2020 (pandemia)
fig.add_vrect(x0=2019.5, x1=2020.5, fillcolor="gray", opacity=0.1,
              annotation_text="COVID", annotation_position="top left")

fig.update_layout(
    title="Vantagem do Mandante ao Longo dos Anos<br><sup>Brasileirão Série A (2003-presente)</sup>",
    xaxis_title="Ano",
    yaxis_title="Percentual (%)",
    template="plotly_white",
    width=900,
    height=500,
    legend=dict(x=0.7, y=0.95)
)

fig.show()
fig.write_html("../charts/mandante_visitante.html", include_plotlyjs="cdn")
print("Gráfico exportado: charts/mandante_visitante.html")

Gráfico exportado: charts/mandante_visitante.html


In [3]:
# Top times com melhor desempenho como mandante
mandante_stats = df.groupby("mandante").apply(
    lambda g: pd.Series({
        "jogos": len(g),
        "vitorias": (g["mandante_Placar"] > g["visitante_Placar"]).sum(),
        "aproveitamento": (g["mandante_Placar"] > g["visitante_Placar"]).mean() * 100
    })
).reset_index()
mandante_stats.columns = ["time", "jogos", "vitorias", "aproveitamento"]

# Filtrar times com pelo menos 100 jogos como mandante
top_mandante = mandante_stats[mandante_stats["jogos"] >= 100].sort_values("aproveitamento", ascending=True).tail(15)

fig2 = px.bar(top_mandante, x="aproveitamento", y="time", orientation="h",
              title="Top 15 Times com Melhor Aproveitamento como Mandante<br><sup>Mínimo 100 jogos em casa (2003-presente)</sup>",
              labels={"aproveitamento": "% de Vitórias em Casa", "time": ""},
              color="aproveitamento",
              color_continuous_scale="Greens",
              text=top_mandante["aproveitamento"].round(1))
fig2.update_layout(template="plotly_white", width=800, height=550, showlegend=False)
fig2.update_traces(textposition="outside")

fig2.show()
fig2.write_html("../charts/top_mandantes.html", include_plotlyjs="cdn")
print("Gráfico exportado: charts/top_mandantes.html")

Gráfico exportado: charts/top_mandantes.html


In [4]:
# Média geral
media_mandante = (df["mandante_Placar"] > df["visitante_Placar"]).mean() * 100
media_empate = (df["mandante_Placar"] == df["visitante_Placar"]).mean() * 100
media_visitante = (df["mandante_Placar"] < df["visitante_Placar"]).mean() * 100

print(f"Média geral (2003-presente):")
print(f"  Mandante vence: {media_mandante:.1f}%")
print(f"  Empate: {media_empate:.1f}%")
print(f"  Visitante vence: {media_visitante:.1f}%")

Média geral (2003-presente):
  Mandante vence: 49.6%
  Empate: 26.4%
  Visitante vence: 24.0%
